In [18]:
#!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv

In [162]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


In [165]:
df = pd.read_csv("course_lead_scoring.csv")

In [167]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [169]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [171]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [173]:
categ = list(df.dtypes[df.dtypes=="object"].index)
num= list(df.dtypes[df.dtypes!="object"].index)

In [175]:
df[num].head()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,88160.0,3,4,0.64,1
1,72688.0,3,7,0.72,1
2,44697.0,3,4,0.58,1
3,NaN,3,6,0.57,0
4,24062.0,4,5,0.62,1


In [177]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [179]:
for c in categ:
    df[c] = df[c].str.lower().str.replace(' ', '_').fillna('NA')

In [181]:
df[categ].head()

,lead_source,industry,employment_status,location
0,organic_search,technology,employed,europe
1,social_media,technology,employed,south_america
2,referral,retail,employed,europe
3,paid_ads,manufacturing,student,NA
4,referral,manufacturing,student,north_america


In [183]:
for n in num:
    df[n] = df[n].fillna(0)

In [185]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1 - retail

In [188]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## Question 2

In [191]:
df[num].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [193]:
df[num].corr().loc[ "interaction_count", "lead_score"]

0.9157457980418697

In [195]:
df[num].corr().loc[ "number_of_courses_viewed", "lead_score"]

0.7572042392104886

In [197]:
df[num].corr().loc[ "number_of_courses_viewed", "interaction_count"]

0.7216090038937035

In [199]:
df[num].corr().loc[ "annual_income", "interaction_count"]

0.12284235135955959

In [201]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42) 
df_full_train.shape, df_test.shape

((4000, 9), (1000, 9))

In [203]:
df_train, df_val = train_test_split(df_full_train, test_size=0.2/0.8, random_state=42)
df_train.shape, df_val.shape, df_test.shape

((3000, 9), (1000, 9), (1000, 9))

In [205]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

##  Question 3 

In [208]:
mscore={}
for c in categ:
    mscore[c]= round(mutual_info_score(df_train.converted, df_train[c]), 2)

In [210]:
mscore

{'lead_source': 0.03,
 'industry': 0.0,
 'employment_status': 0.02,
 'location': 0.0}

## Question 4

In [213]:
del df_train['converted']
del df_val['converted']
del df_test['converted']
del df_full_train['converted']

In [215]:
train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')

In [217]:
dv = DictVectorizer(sparse = False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

In [219]:
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

C:\Users\Admin-IT\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=42)

In [221]:
y_val_pro = model.predict_proba(X_val)[:, 1]
y_val_predict = model.predict(X_val)

In [223]:
co_dec = y_val_predict

In [227]:
round((y_val == co_dec.astype(int)).mean(), 2)

0.72

In [229]:
(y_val == co_dec.astype(int)).mean()

0.722

## Question 5

In [243]:
acc_score = {}
for remove in ['industry', 'employment_status', 'lead_score']:
    train_dict_r = df_train.drop([remove], axis=1).to_dict(orient='records')
    val_dict_r = df_val.drop([remove], axis=1).to_dict(orient='records')
    test_dict_r = df_test.drop([remove], axis=1).to_dict(orient='records')
    dv = DictVectorizer(sparse = False)
    X_train_r = dv.fit_transform(train_dict_r)
    X_val_r = dv.fit_transform(val_dict_r)
    X_test_r = dv.fit_transform(test_dict_r)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_r, y_train)
    y_val_pro_r = model.predict_proba(X_val_r)[:, 1]
    co_dec = ( y_val_pro_r >= 0.5 )
    acc_score[remove] = (y_val == co_dec.astype(int)).mean()


In [247]:
acc_score

{'industry': 0.645, 'employment_status': 0.645, 'lead_score': 0.644}

In [249]:
orig_acc = (y_val == conv_decision.astype(int)).mean()
orig_acc

0.645

In [251]:
for remove in ['industry', 'employment_status', 'lead_score']:
    diff = orig_acc - acc_score[remove]
    print(remove, diff)

industry 0.0
employment_status 0.0
lead_score 0.0010000000000000009


## Question 6

In [253]:
acc_score = {}
for c_value in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', C=c_value, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_val_pro_c = model.predict_proba(X_val)[:, 1]
    co_dec = ( y_val_pro_c >= 0.5 )
    acc_score[str(c_value)] = (y_val == co_dec.astype(int)).mean()

acc_score

{'0.01': 0.645, '0.1': 0.645, '1': 0.645, '10': 0.645, '100': 0.645}